# Reasoning-Gym · SFT warm-start → GRPO → evaluate — Hugging Face Jobs

_Authored by: [Behrooz Azarkhalili](https://github.com/behroozazarkhalili)_

> **An OpenEnv agent-training walkthrough, HF Jobs edition.** This notebook is meant to
> be executed **non-interactively on Hugging Face infrastructure** with `papermill`. The
> GPU work runs on HF's cloud; you only submit.

**What this trains:** a small LM to act as an agent inside a live OpenEnv environment via
**GRPO** (chain-sum arithmetic), with a short **SFT warm-start**, then a held-out evaluation.

### What you'll learn

- Submit a notebook to run **non-interactively on Hugging Face Jobs** with `papermill` — the GPU work runs on HF's cloud, you only submit
- Drive a live [**OpenEnv**](https://github.com/huggingface/OpenEnv) environment: collect teacher rollouts → filter the correct ones → serialize them to a dataset
- Warm-start a policy with a short **SFT** pass on the correct rollouts, using assistant-only loss masking so the model learns to *emit* the tool call
- Train the agent with **GRPO** — value-free RL where the environment's scalar reward is the only training signal (no separate reward model)
- Evaluate honestly: run **base vs. trained** on a held-out seed and report accuracy and format compliance

## ▶ Submit this notebook as an HF Job

Requires a positive [credit balance](https://huggingface.co/settings/billing) on your HF account (Jobs are pay-as-you-go — you pay only for the seconds you use). The control plane is plain HTTPS; the recipe below clones this notebook straight from
the cookbook repo, so it runs as-is.

```bash
hf jobs run \
  --flavor a10g-small \
  --timeout 10800 \
  --secrets HF_TOKEN \
  -e SMOKE=0 -e REPORT_TO=trackio \
  -e ENV_BASE_URL=https://sergiopaniego-reasoning-gym.hf.space \
  python:3.12 \
  bash -c "apt-get update -q && apt-get install -y -q git && \
           pip install -q papermill && \
           pip install -q trl openenv 'transformers>=5.3.0' trackio openai jmespath nest_asyncio datasets && \
           pip install -q --no-deps git+https://huggingface.co/spaces/sergiopaniego/reasoning_gym && \
           git clone --depth 1 https://github.com/huggingface/cookbook /cookbook && cd /cookbook/notebooks/en && \
           papermill grpo_agent_reasoning_gym_hf_jobs.ipynb out.ipynb"
```

`--secrets HF_TOKEN` forwards your token so the trained model can push to the Hub. Set
`SMOKE=0` for a real run (150 GRPO steps) or `SMOKE=1` for a quick smoke. Tune `--flavor`
(`t4-small`, `a10g-small`, `a100-large`, …) and `--timeout` (seconds) to your run.

### Faster rollouts with vLLM

An HF Job is a non-interactive runtime, so vLLM-accelerated generation works here (it is
left off in interactive notebooks because its init conflicts with IPython). Use a bigger
GPU and install vLLM in the same command:

```bash
hf jobs run \
  --flavor a100-large \
  --timeout 21600 \
  --secrets HF_TOKEN \
  -e SMOKE=0 -e USE_VLLM=1 -e REPORT_TO=trackio \
  -e ENV_BASE_URL=https://sergiopaniego-reasoning-gym.hf.space \
  python:3.12 \
  bash -c "apt-get update -q && apt-get install -y -q git && \
           pip install -q papermill vllm && \
           pip install -q trl openenv 'transformers>=5.3.0' trackio openai jmespath nest_asyncio datasets && \
           pip install -q --no-deps git+https://huggingface.co/spaces/sergiopaniego/reasoning_gym && \
           git clone --depth 1 https://github.com/huggingface/cookbook /cookbook && cd /cookbook/notebooks/en && \
           papermill grpo_agent_reasoning_gym_hf_jobs.ipynb out.ipynb"
```

## 0 · Install

We install TRL (the trainer), OpenEnv (the env client), and the environment's own package
(it ships from its Space). `transformers>=5.3.0` is required because GRPO's
`environment_factory` path — a TRL feature — depends on tool-calling / chat-template
behavior introduced in that Transformers release. `trackio` gives live training charts;
`openai` is only needed if you choose the OpenAI teacher in the collect step (optional).

In [ ]:
%pip install -q trl openenv "transformers>=5.3.0" trackio openai jmespath nest_asyncio datasets
%pip install -q --no-deps git+https://huggingface.co/spaces/sergiopaniego/reasoning_gym

### Why `nest_asyncio`?

The OpenEnv client is **async** (`await env.reset()`, `await env.step()`), but a Jupyter/Colab kernel already runs its own event loop. `nest_asyncio.apply()` lets us `await` inside notebook cells without `RuntimeError: event loop already running`.

> 💡 In a plain `.py` script you'd use `asyncio.run(...)` instead — this shim is specifically for notebook runtimes.

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
# --- Authenticate with the Hugging Face Hub (portable) -------------------------
# Prefers an already-set HF_TOKEN (HF Jobs / Colab secret); falls back to the
# interactive widget. Never hard-codes a token.
import os

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"])
    print("Authenticated via HF_TOKEN.")
else:
    try:
        from huggingface_hub import notebook_login

        notebook_login()
    except Exception as exc:  # non-interactive without a token
        print(f"notebook_login unavailable ({exc}); set HF_TOKEN before running.")

### Resolve your Hub username

Downstream repo names (rollouts dataset, trained model) are built from your username, so we resolve it once via `whoami()` rather than hard-coding it. This keeps the notebook portable across accounts.

> 💡 `whoami()` reads the token you authenticated with above — no extra prompt.

In [ ]:
# Resolve your Hub username automatically — no hard-coded usernames downstream.
from huggingface_hub import whoami

try:
    HF_USERNAME = whoami()["name"]
    print(f"Hub user: {HF_USERNAME}")
except Exception as exc:
    HF_USERNAME = os.environ.get("HF_USERNAME", "")
    print(f"Could not resolve whoami() ({exc}); using HF_USERNAME={HF_USERNAME!r}.")

### Auto-detect compute, pick a model that fits

Rather than hard-code a model, we read the GPU's VRAM and choose a size that fits (`Qwen3-1.7B` when there's headroom, else `Qwen3-0.6B`). Override with `MODEL_NAME`. This is what lets the same notebook run on a T4, an L4, or an A100 unchanged.

> 💡 GRPO holds the policy model **and** generates rollouts, so VRAM headroom matters more than for plain inference. [TRL GRPO docs](https://huggingface.co/docs/trl/en/grpo_trainer).

In [ ]:
# --- Auto-detect compute + pick a model that fits ------------------------------
# Larger model only when there is VRAM headroom; otherwise fall back. You can
# override by setting MODEL_NAME in the environment before launching.
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    DEVICE = "cuda"
else:
    vram_gb = 0.0
    DEVICE = "cpu"

DEFAULT_MODEL = "Qwen/Qwen3-1.7B" if vram_gb >= 24 else "Qwen/Qwen3-0.6B"
MODEL_NAME = os.environ.get("MODEL_NAME", DEFAULT_MODEL)
print(f"device={DEVICE}  vram={vram_gb:.1f} GB  ->  MODEL_NAME={MODEL_NAME}")

### Run knobs: smoke vs. full

Every expensive quantity (rollout count, GRPO steps, eval size) is gated by `SMOKE` so you can prove the whole pipeline end-to-end in minutes (`SMOKE=1`) before committing to a real run (`SMOKE=0`). The values come from environment variables, so you never edit cells to change a run.

> 💡 This is the single switch that turns a 5-minute demo into a real training run.

In [ ]:
# --- Run knobs (smoke vs. full) ------------------------------------------------
# SMOKE keeps everything tiny so the whole notebook runs in minutes on one GPU to
# prove the pipeline end-to-end. Set SMOKE=0 (or env SMOKE=0) for a real run.
SMOKE = os.environ.get("SMOKE", "1") not in ("0", "false", "False")

ENV_BASE_URL = os.environ.get("ENV_BASE_URL", "https://sergiopaniego-reasoning-gym.hf.space")
DATASET_NAME = "chain_sum"
DATASET_CONFIG = {"min_terms": 2, "max_terms": 3, "min_digits": 2, "max_digits": 2}

N_EPISODES_COLLECT = 30 if SMOKE else 300     # teacher rollouts for SFT
GRPO_MAX_STEPS = 5 if SMOKE else 150          # GRPO optimisation steps
N_EVAL = 10 if SMOKE else 50                  # held-out eval problems
print(f"SMOKE={SMOKE}  collect={N_EPISODES_COLLECT}  grpo_steps={GRPO_MAX_STEPS}  eval={N_EVAL}")
print(f"env={ENV_BASE_URL}")

## 1 · (Optional) Deploy your own environment Space

Hosted tutorial Spaces are **low-concurrency** — fine for this notebook, a bottleneck
for a real run where the trainer spins up many parallel envs. For production, duplicate
the Space to your own account (`openenv push`, or the Hub "Duplicate this Space" button)
and set `ENV_BASE_URL` to it. Left as a documented step so the notebook stays runnable
out-of-the-box against the hosted default.

In [ ]:
# Uncomment to deploy your own copy (requires Docker locally OR use the Hub UI button):
#   !openenv push sergiopaniego/reasoning_gym --to {HF_USERNAME}/reasoning_gym
# then:  ENV_BASE_URL = f"https://{HF_USERNAME.replace('_','-')}-reasoning-gym.hf.space"
print("Using ENV_BASE_URL =", ENV_BASE_URL)

## 2 · Phase A — collect teacher rollouts (the "dataset")

OpenEnv's training data is **generated**, not downloaded: a **teacher** model acts in the
env with the OpenEnv *harness*, which records each episode's messages and the env's reward.
This is the real OpenEnv data-generation step — `CollectRunner` drives the teacher,
`RolloutSerializer` writes episodes to disk, `push_to_hf_hub` publishes them as a dataset.
We keep the episodes the teacher got *right*; those become the SFT warm-start set.

The teacher is a **parameter**. The tutorial uses OpenAI `gpt-5-mini`; for a fully-open,
no-API-key path you can point `create_llm_client` at any OpenAI-compatible endpoint
(e.g. a TGI/vLLM server) via `TEACHER_PROVIDER` / `TEACHER_MODEL`. If no teacher key is
set, we skip straight to GRPO — the notebook still runs end-to-end.

In [ ]:
SYSTEM_PROMPT = """You are a careful arithmetic assistant.

You will be given a chain of integer additions. Compute the result and submit it as a single number.

Rules:
1. Read the question carefully.
2. Use the tool `answer` exactly once with your final number.
3. The answer must be a single integer with no units or explanation.
"""

### The teacher (optional)

GRPO trains best from a warm-started policy. To create that warm-start we let a stronger **teacher** model play the env and keep its *correct* episodes for SFT. The teacher is a parameter: an OpenAI model, any OpenAI-compatible endpoint, or — if none is set — we skip collection and let GRPO cold-start, so the notebook always runs.

> 💡 No API key? Leave it unset; you'll still get a complete GRPO run, just without the SFT warm-start.

In [ ]:
# Teacher is configurable. Tutorial-faithful default is OpenAI gpt-5-mini, BUT a teacher
# requires a reachable endpoint: an OpenAI key, OR a self-hosted OpenAI-compatible URL.
# If neither is available we cannot collect — fail LOUD and early with a clear message
# rather than deep inside CollectRunner. (Set TEACHER_BASE_URL to a TGI/vLLM server for a
# fully-open, no-key path.)
TEACHER_PROVIDER = os.environ.get("TEACHER_PROVIDER", "openai")
TEACHER_MODEL = os.environ.get("TEACHER_MODEL", "gpt-5-mini")
TEACHER_BASE_URL = os.environ.get("TEACHER_BASE_URL")  # set for a self-hosted endpoint

_has_openai_key = bool(os.environ.get("OPENAI_API_KEY"))
TEACHER_AVAILABLE = bool(TEACHER_BASE_URL) or (TEACHER_PROVIDER == "openai" and _has_openai_key)
if not TEACHER_AVAILABLE:
    print(
        "[skip-collect] No teacher endpoint available "
        "(set OPENAI_API_KEY, or TEACHER_BASE_URL for an OpenAI-compatible server).\n"
        "Phase A (collect+SFT) will be SKIPPED; GRPO will cold-start from the base model.\n"
        "This keeps the notebook runnable end-to-end with zero external keys."
    )

In [ ]:
# Collect only if a teacher is reachable. Otherwise we skip straight to GRPO (cold-start),
# so the notebook always runs top-to-bottom.
RUN_SFT = TEACHER_AVAILABLE
correct = []  # populated by Phase A when it runs

if TEACHER_AVAILABLE:
    from openenv.core.harness import HarnessRunLimits, MCPHarnessAdapter
    from openenv.core.harness.collect import (
        CollectRunner,
        RolloutSerializer,
        build_model_step,
        push_to_hf_hub,
    )
    from openenv.core.llm_client import create_llm_client
    from reasoning_gym_env.client import ReasoningGymEnv
    from reasoning_gym_env.harness import ReasoningGymSessionFactory

    _client_kwargs = {"provider": TEACHER_PROVIDER, "model": TEACHER_MODEL, "max_tokens": 1024}
    if TEACHER_PROVIDER == "openai" and _has_openai_key:
        _client_kwargs["api_key"] = os.environ["OPENAI_API_KEY"]
    if TEACHER_BASE_URL:
        _client_kwargs["base_url"] = TEACHER_BASE_URL

    llm_client = create_llm_client(**_client_kwargs)
    model_step = build_model_step(llm_client, system_prompt=SYSTEM_PROMPT)

    factory = ReasoningGymSessionFactory(
        lambda: ReasoningGymEnv(base_url=ENV_BASE_URL),
        dataset_name=DATASET_NAME,
        dataset_config=DATASET_CONFIG,
    )
    serializer = RolloutSerializer("./rollouts")
    runner = CollectRunner(
        session_factory=factory,
        harness_adapter=MCPHarnessAdapter(),
        serializer=serializer,
        limits=HarnessRunLimits(max_turns=9),
    )
    result = runner.run(model_step=model_step, num_episodes=N_EPISODES_COLLECT)
    print(
        f"collected={result.num_collected} dropped={result.num_dropped} "
        f"avg_reward={result.avg_reward:.3f} success_rate={result.success_rate:.0%}"
    )

In [ ]:
# Publish the rollouts to the Hub only when authenticated (optional, for sharing/reuse).
# The SFT step reads from the LOCAL ./rollouts dir, so a Hub push is never required to run.
if TEACHER_AVAILABLE and HF_USERNAME:
    try:
        url = push_to_hf_hub(output_dir="./rollouts", repo_id=f"{HF_USERNAME}/chain-sum-rollouts")
        print(f"Rollouts dataset: {url}")
    except Exception as exc:
        print(f"[warn] Hub push skipped ({exc}); continuing from local ./rollouts.")

## 3 · Filter + format the rollouts for SFT

Keep only episodes where the teacher was **correct** (`reward == 1.0`), and rewrite the
assistant tool call into the `<tool_call>{json}</tool_call>` text form the student learns
to emit. We strip the env's `tool` responses — SFT only supervises the assistant turn.
We read rollouts from the **local** `./rollouts` directory (no Hub round-trip needed).

In [ ]:
import json

from datasets import load_dataset

if RUN_SFT:
    # Load from local disk — portable, no Hub auth required.
    ds = load_dataset("json", data_files="./rollouts/*.jsonl", split="train")
    raw_rollouts = list(ds)
    print(f"loaded {len(raw_rollouts)} episodes from ./rollouts")
else:
    raw_rollouts = []
    print("Phase A skipped (no teacher) — no rollouts to format.")


def to_chat_messages(record):
    converted = []
    for msg in record["messages"]:
        if msg["role"] == "tool":
            continue  # SFT supervises only the assistant turn
        if msg["role"] == "assistant" and msg.get("tool_calls"):
            tc = msg["tool_calls"][0]
            args = json.loads(tc["function"]["arguments"])
            tool_call_text = (
                "<tool_call>\n"
                + json.dumps({"name": "answer", "arguments": {"answer": args.get("answer", "")}})
                + "\n</tool_call>"
            )
            converted.append({"role": "assistant", "content": tool_call_text})
        else:
            converted.append(msg)
    return {"messages": converted, "reward": record["reward"]}


import re as _re


def _slug(s: str) -> str:
    # HF/trackio Space ids prefer hyphen-lowercase; underscores get normalised in the
    # Space subdomain, so build a clean slug up front (matches the canonical naming).
    return _re.sub(r"-+", "-", _re.sub(r"[^a-z0-9]+", "-", s.lower())).strip("-")


_RUN_TAG = _slug(f"reasoning-gym-{DATASET_NAME}-{MODEL_NAME.split('/')[-1]}")
SFT_OUT = f"{_RUN_TAG}-sft"

rollouts = [to_chat_messages(r) for r in raw_rollouts]
correct = [r for r in rollouts if r["reward"] == 1.0]
if RUN_SFT:
    print(f"correct: {len(correct)} / {len(rollouts)} ({len(correct) / max(len(rollouts),1):.1%})")
    if not correct:
        # No usable teacher rollouts -> skip SFT rather than abort; GRPO cold-starts.
        RUN_SFT = False
        print("[skip-sft] 0 correct rollouts; GRPO will cold-start from the base model.")

## 4 · Size the sequence length from the data

When SFT runs, rather than guess `max_length` we measure it: tokenize the kept rollouts
and set the cap at the 99th percentile + a small margin. Data-driven, model-agnostic.

In [ ]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_SEQ_LEN = 1024  # default if SFT is skipped
if RUN_SFT and correct:
    lengths = []
    for row in correct:
        text = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)
        lengths.append(len(tokenizer.encode(text)))
    lengths = np.array(lengths)
    MAX_SEQ_LEN = int(np.percentile(lengths, 99)) + 16
    print(f"p50={np.percentile(lengths,50):.0f} p95={np.percentile(lengths,95):.0f} "
          f"p99={np.percentile(lengths,99):.0f} max={lengths.max()}  -> MAX_SEQ_LEN={MAX_SEQ_LEN}")
else:
    print(f"SFT skipped; MAX_SEQ_LEN default = {MAX_SEQ_LEN}")

## 5 · Phase A — SFT warm-start

A few epochs of supervised fine-tuning on the *correct* rollouts teach the model the env's
tool-call format **before** RL. `assistant_only_loss=True` masks the loss to the assistant
tokens, so the student learns *to produce the tool call*, not to parrot the prompt. The
result is a checkpoint that already calls the `answer` tool in the right format — a more
prepared, more stable GRPO starting point than the base model. **Runs only when Phase A
collected usable rollouts**; otherwise GRPO cold-starts from the base model and the
notebook continues.

In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

if RUN_SFT and correct:
    sft_dataset = Dataset.from_list([{"messages": r["messages"]} for r in correct])
    sft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    sft_config = SFTConfig(
        output_dir=SFT_OUT,
        hub_model_id=f"{HF_USERNAME}/{SFT_OUT}-model" if HF_USERNAME else f"{SFT_OUT}-model",
        max_length=MAX_SEQ_LEN,
        num_train_epochs=1 if SMOKE else 3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="no",
        assistant_only_loss=True,
        push_to_hub=not SMOKE,
    )
    sft_trainer = SFTTrainer(
        model=sft_model,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
        args=sft_config,
    )
    sft_trainer.train()
    sft_trainer.save_model(SFT_OUT)
    if not SMOKE:
        sft_trainer.push_to_hub(commit_message="SFT warm-up on reasoning_gym chain_sum")
    print(f"SFT checkpoint -> {SFT_OUT}")
else:
    print("SFT skipped — GRPO will cold-start from", MODEL_NAME)

## 6 · Phase B — GRPO

Now the RL phase. GRPO is **value-free**: for each prompt it samples a group of rollouts,
scores each by the env's reward, and pushes the policy toward the better ones *within* the
group — the environment's reward, not a static label, is the only training signal. More
`num_generations` gives a richer in-group ranking but costs more generation. The pieces,
all verified from the canonical walkthrough:

- **An env wrapper class.** Its public, documented methods become the agent's *tools*.
  Here `answer(...)` is the single tool; `reset()` pulls the next question. The class
  tracks `self.reward` / `self.done` so the trainer can read the outcome.
- **A reward function** that just reads `env.reward` off each rollout's env instance.
- **A dummy prompt dataset** whose only job is to set the episode count — the *real*
  prompts come from `env.reset()` inside the factory.
- **`environment_factory=<class>`** on `GRPOTrainer`: the trainer creates one env per
  rollout, generates the completion, parses the tool call, steps the env, and collects
  the reward — the entire agent loop, automated.

In [ ]:
import random

from reasoning_gym_env import ReasoningGymAction, ReasoningGymEnv


class ReasoningGymTrainEnv:
    """One rollout = one question -> one `answer` tool call -> done."""

    DATASET_SIZE = 1000

    def __init__(self):
        self.client = ReasoningGymEnv(base_url=ENV_BASE_URL).sync()
        self._dataset_seed = random.randint(0, 2**31 - 1)  # per-instance: parallel envs diverge
        self._initialized = False
        self.reward = 0.0
        self.done = False

    def reset(self, **kwargs) -> str:
        if not self._initialized:
            result = self.client.reset(
                dataset_name=DATASET_NAME,
                dataset_config=DATASET_CONFIG,
                seed=self._dataset_seed,
                size=self.DATASET_SIZE,
            )
            self._initialized = True
        else:
            result = self.client.reset()  # next question; re-sending config would rewind to 0
        self.reward = 0.0
        self.done = False
        return result.observation.question

    def answer(self, answer: str) -> str:
        """Submit the final answer for the current question.

        Args:
            answer: The agent's answer (parsed as a number server-side).

        Returns:
            Feedback string with the score and the correct answer.
        """
        if self.done:
            raise ValueError("Episode is already finished.")
        result = self.client.step(ReasoningGymAction(answer=str(answer)))
        self.reward = float(result.observation.score or 0.0)
        self.done = True
        return f"score={self.reward} correct={result.observation.correct_answer}"


def reward_func(environments, **kwargs) -> list[float]:
    return [env.reward for env in environments]

In [ ]:
from datasets import Dataset

# Dummy prompts: count == number of rollout episodes. Real prompts come from reset().
N_PROMPTS = 50 if SMOKE else 1000
grpo_dataset = Dataset.from_dict(
    {"prompt": [[{"role": "user", "content": SYSTEM_PROMPT}] for _ in range(N_PROMPTS)]}
)

### Configure GRPO

`GRPOConfig` holds the RL hyperparameters. The agent-specific ones: `num_generations` (rollouts sampled per prompt — GRPO ranks them against each other), `max_completion_length` (room for the tool call), and `report_to="trackio"` for live charts. `save_strategy`/`push_to_hub` control persistence.

> 💡 GRPO is *value-free*: it needs no separate reward model — the **environment's** scalar reward is the only signal. [GRPO paper](https://huggingface.co/papers/2402.03300).

In [ ]:
from trl import GRPOConfig

GRPO_OUT = f"{_RUN_TAG}-grpo"

# Logging backend is a knob: "trackio" (default, live charts) or "none" for a fully
# offline/headless run with no Space setup. trackio_space_id is only set when using trackio.
REPORT_TO = os.environ.get("REPORT_TO", "trackio")
_report_kwargs = {"report_to": REPORT_TO}
if REPORT_TO == "trackio":
    _report_kwargs["trackio_space_id"] = GRPO_OUT

grpo_config = GRPOConfig(
    num_train_epochs=1,
    max_steps=GRPO_MAX_STEPS,
    learning_rate=1e-6,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=1,
    warmup_steps=min(10, GRPO_MAX_STEPS),
    optim="adamw_torch",
    max_grad_norm=1.0,
    num_generations=2,
    max_completion_length=256,
    log_completions=True,
    num_completions_to_print=2,
    chat_template_kwargs={"enable_thinking": False},
    output_dir=GRPO_OUT,
    # Push weights to a repo DISTINCT from the Trackio Space id (=GRPO_OUT),
    # otherwise push_to_hub sees the Trackio-owned repo and skips as 'no files
    # modified'. A separate -model repo gets the actual checkpoint commit.
    hub_model_id=f"{HF_USERNAME}/{GRPO_OUT}-model" if HF_USERNAME else f"{GRPO_OUT}-model",
    logging_steps=1 if SMOKE else 10,
    gradient_checkpointing=True,
    save_strategy="no",
    push_to_hub=not SMOKE,
    **_report_kwargs,
    # vLLM left OFF in-notebook: its init breaks under IPython. The optional cell below
    # enables it (use_vllm=True, vllm_mode="colocate") for an HF Job / script run.
)

In [ ]:
# --- OPTIONAL: vLLM-accelerated GRPO rollouts (5-10x faster generation) ---------
# OFF by default: vLLM's init breaks under IPython/Jupyter, so leave USE_VLLM=0 for an
# in-notebook run. Enable it (USE_VLLM=1) ONLY in a non-IPython context: an HF Job
# or a fresh Colab runtime executed as a script. Colocate mode shares
# the single training GPU (right for Colab / one-GPU HF-Job flavors).
#   Verified TRL v1.7.0 params: use_vllm, vllm_mode="colocate"|"server",
#   vllm_gpu_memory_utilization. (server mode is multi-GPU; not used here.)
import os

USE_VLLM = os.environ.get("USE_VLLM", "0") not in ("0", "false", "False")
if USE_VLLM:
    grpo_config.use_vllm = True
    grpo_config.vllm_mode = "colocate"
    # Leave room for the training copy of the model in colocate mode; tune per GPU/model.
    grpo_config.vllm_gpu_memory_utilization = float(
        os.environ.get("VLLM_GPU_MEM_UTIL", "0.3")
    )
    print(
        f"[vLLM] enabled: mode=colocate gpu_mem_util={grpo_config.vllm_gpu_memory_utilization} "
        "(requires a non-IPython runtime + `pip install vllm`)"
    )
else:
    print("[vLLM] disabled (USE_VLLM=0). In-notebook generation uses HF generate().")

### Train the agent

`environment_factory=<EnvClass>` is the key line: for each rollout the trainer creates an env (TRL may reuse env instances across a batch), generates the model's response, parses its tool call, steps the env, and reads the reward — the agent loop, automated.

We warm-start from this run's SFT checkpoint when one exists, else from the base model.

> 💡 This is what makes it *agent* training rather than text fine-tuning: the data is generated by the policy acting in the env, not read from a file.

In [ ]:
from trl import GRPOTrainer

# Warm-start GRPO from the SFT checkpoint when available; else from base.
import os as _os

# Warm-start from THIS run's SFT checkpoint only. Guard on RUN_SFT so a stale SFT dir
# left in the cwd by a PRIOR run does not silently warm-start a cold-start run.
GRPO_INIT = SFT_OUT if (RUN_SFT and _os.path.isdir(SFT_OUT)) else MODEL_NAME
print(f"GRPO initialising from: {GRPO_INIT}")

grpo_trainer = GRPOTrainer(
    model=GRPO_INIT,
    reward_funcs=reward_func,
    train_dataset=grpo_dataset,
    args=grpo_config,
    environment_factory=ReasoningGymTrainEnv,
)
grpo_trainer.train()
grpo_trainer.save_model(GRPO_OUT)
if not SMOKE:
    grpo_trainer.push_to_hub(commit_message="GRPO fine-tune on reasoning_gym chain_sum")

## 7 · Training signal — reward delta

The quickest sanity check: did mean reward rise over training? We read it from the
trainer's log history and compare the mean over the first few logged steps vs. the last
few — a check on whether reward *moved* during training, not a held-out quality claim.

In [ ]:
import statistics

rewards = [log["reward"] for log in grpo_trainer.state.log_history if "reward" in log]
if len(rewards) < 5:
    print(f"Only {len(rewards)} reward logs — increase max_steps / lower logging_steps for a clean delta.")
else:
    initial, final = statistics.mean(rewards[:5]), statistics.mean(rewards[-5:])
    print(f"initial reward (first5): {initial:.2%}")
    print(f"final   reward (last5):  {final:.2%}")
    print(f"delta:                   {(final - initial) * 100:+.2f} pp")

## 8 · Test the trained agent against held-out problems

The question that matters: does the agent solve problems it never trained on? We run both
the base model and the GRPO checkpoint on a held-out `seed` (a disjoint set of problems),
with identical decoding settings so the comparison is fair, and report two numbers per model:

- **accuracy** — fraction of held-out problems the env scores correct;
- **format compliance** — fraction where the model emitted a parseable `answer`.

This is the reference `evaluate_model` harness from the HF SFT-warmup tutorial.

In [ ]:
import re

from transformers import pipeline
from reasoning_gym_env.client import ReasoningGymEnv
from reasoning_gym_env.models import ReasoningGymAction


async def evaluate_model(model_name_or_path, n_eval=N_EVAL, seed=999):
    """Held-out accuracy + format-compliance for one model (reference strategy).

    Single-turn: render [system, user], generate once, extract the answer, step the env
    once, read the env score. `format_compliance` = fraction emitting a parseable
    `"answer": <int>` (the trained tool-call surface). Higher seed => unseen problems.
    """
    gen = pipeline(
        "text-generation",
        model=model_name_or_path,
        tokenizer=model_name_or_path,
        device_map="auto",
        dtype="auto",
    )
    gen.model.generation_config.max_length = None
    tok = AutoTokenizer.from_pretrained(model_name_or_path)
    eval_env = ReasoningGymEnv(base_url=ENV_BASE_URL)

    obs = await eval_env.reset(
        dataset_name=DATASET_NAME, dataset_config=DATASET_CONFIG, seed=seed, size=n_eval
    )

    rewards, format_hits = [], 0
    for i in range(n_eval):
        if i:
            obs = await eval_env.reset()
        question = obs.observation.question
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        completion = gen(prompt, max_new_tokens=256)[0]["generated_text"][len(prompt):]

        # Reference extraction: `"answer": <int>` sets format compliance; else last integer.
        # Signed ints kept (our param): chain_sum results can be negative.
        m = re.search(r'"answer"\s*:\s*"?(-?\d+)"?', completion)
        if m:
            format_hits += 1
            answer = m.group(1)
        else:
            nums = re.findall(r"(-?\d+)", completion)
            answer = nums[-1] if nums else "0"

        res = await eval_env.step(ReasoningGymAction(answer=answer))
        rewards.append(float(res.observation.score or 0.0))

    await eval_env.close()
    del gen
    return {"accuracy": sum(rewards) / len(rewards), "format_compliance": format_hits / n_eval}


### Base vs. trained, on held-out problems

We run the **same** seeded held-out set through the base model and the trained one and compare. Using a different seed than training guarantees the problems are unseen; using the same seed for both models makes the comparison apples-to-apples.

> 💡 Training reward rising is necessary but not sufficient — this held-out check is the honest measure of whether the agent generalized.

In [ ]:
# Compare base vs. the GRPO-trained checkpoint on held-out problems.
base_metrics = await evaluate_model(MODEL_NAME)
trained_metrics = await evaluate_model(GRPO_OUT)

print(f"\n{'Metric':<22}{'Base':>10}{'Trained':>10}{'Delta':>10}")
print("-" * 52)
for key, label in [("format_compliance", "Format compliance"), ("accuracy", "Accuracy")]:
    b, t = base_metrics[key], trained_metrics[key]
    print(f"{label:<22}{b:>10.1%}{t:>10.1%}{(t - b) * 100:>+9.1f}pp")


## Recap

You ran the complete OpenEnv agent-training loop: **collect → filter → SFT warm-start →
GRPO → held-out evaluation**, all against a live environment, all portable. The trained
agent's quality is measured on accuracy on problems it never saw.

- For a real run: set `SMOKE=0`, optionally deploy your own env Space, and enable
  vLLM via `USE_VLLM=1` on an HF Job (see the run-recipe cell near the top).
- **Next:** [the Wordle notebook](https://huggingface.co/learn/cookbook/grpo_agent_wordle_hf_jobs) applies the
  same shape to a genuinely *multi-turn* agentic environment (6 guesses per episode with
  stateful feedback).

## References

### Papers and Research
- **GRPO Algorithm**: [Group Relative Policy Optimization](https://huggingface.co/papers/2402.03300) — the original GRPO paper (DeepSeekMath), introducing value-free, group-relative policy optimization
- **RLHF foundations**: [Fine-Tuning Language Models from Human Preferences](https://arxiv.org/abs/1909.08593) — Ziegler et al.; the reward-driven fine-tuning lineage GRPO belongs to

### Libraries and Frameworks
- **TRL (Transformers Reinforcement Learning)**: [huggingface/trl](https://github.com/huggingface/trl) — the `GRPOTrainer` and `environment_factory` agent-training path used here · [TRL GRPO docs](https://huggingface.co/docs/trl/en/grpo_trainer) · [TRL OpenEnv guide](https://huggingface.co/docs/trl/en/openenv)
- **OpenEnv**: [huggingface/OpenEnv](https://github.com/huggingface/OpenEnv) — the environment client, harness, and rollout collection API · [OpenEnv announcement](https://huggingface.co/blog/openenv)
- **Transformers**: [huggingface/transformers](https://github.com/huggingface/transformers) — `>=5.3.0` required for the tool-calling / chat-template behavior GRPO's env path depends on
- **Hugging Face Jobs**: [running Jobs from the CLI](https://huggingface.co/docs/huggingface_hub/en/guides/cli#hf-jobs) — non-interactive GPU execution on HF infrastructure
- **papermill**: [papermill docs](https://papermill.readthedocs.io/) — parameterized, non-interactive notebook execution
- **Trackio**: [gradio-app/trackio](https://github.com/gradio-app/trackio) — lightweight live training charts

### Environments and Models
- **reasoning_gym environment Space**: [sergiopaniego/reasoning_gym](https://huggingface.co/spaces/sergiopaniego/reasoning_gym) — the hosted OpenEnv environment (chain-sum arithmetic) this notebook trains against
- **Qwen3-0.6B**: [Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) — default policy on smaller GPUs
- **Qwen3-1.7B**: [Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) — default policy when VRAM allows

### Key Concepts
- **Agentic RL**: the training data is *generated* by the policy acting in a live environment, not read from a static file
- **Value-free RL (GRPO)**: no separate reward model or value network — the environment's scalar reward is the only signal; rollouts are ranked *within* a sampled group
- **SFT warm-start**: a short supervised pass on the teacher's *correct* rollouts (assistant-only loss masking) so GRPO begins from a policy that already emits the tool call
- **Held-out generalization**: evaluating on a disjoint seed measures whether the agent solves problems it never trained on — the honest quality signal beyond rising training reward